# Summary

| Method | RMSE | Rank |
|---|---|---|
| Baseline | 0.2145 | 3551/3931 (90.33%) |
| XGB / HPO / Voting | 0.1198 | 273/3931 (6.94%) |
| Log-Ridge | 0.1184 | 157/3931 (3.99%) |
| **Data Pruning** | **0.1180** | **153/3931 (3.89%)** |

# Setup

In [1]:
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi


# Seed
seed = 42

# Kaggle auth
token_path = Path().home() / '.kaggle' / 'access_token'
token = token_path.read_text().strip()
os.environ['KAGGLE_API_TOKEN'] = token

api = KaggleApi()
api.authenticate()

# sklearn config
from sklearn import set_config
set_config(
    transform_output='pandas',
    display='text',
)

# 1. Data

In [2]:
import kagglehub

# Download
path = './data'
if not Path(path).is_dir():
    kagglehub.competition_download(
        handle='house-prices-advanced-regression-techniques',
        output_dir='./data',
    )
else:
    print("Dataset already exists.")

# Load
df_train = pd.read_csv(path + '/train.csv')
X_train  = df_train.drop(columns=['Id', 'SalePrice'])
y_train  = df_train['SalePrice']
df_test  = pd.read_csv(path + '/test.csv')

# Data types
cols_num = X_train.select_dtypes(include='number').columns
cols_cat = X_train.select_dtypes(exclude='number').columns

Dataset already exists.


c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Preprocessing

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import KFold

prep_num = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler(),
)

prep_cat = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    TargetEncoder(
        target_type='continuous',
        cv=KFold(
            n_splits=5,
            shuffle=True,
            random_state=seed,
        ),
    )
)

preprocessing = ColumnTransformer(
    [
        ('pipe_num', prep_num, cols_num),
        ('pipe_cat', prep_cat, cols_cat),
    ]
)

# 3. Model

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    objective='reg:squarederror',
    device='cuda',
    tree_method='hist',
    enable_categorical=False,
    random_state=seed,
)

# 4. HPO

In [ ]:
import optuna
import cupy as cp
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import cross_val_score


def to_gpu(X):
    return cp.asarray(np.asarray(X), dtype=cp.float32)

# 1. Objective
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int(name='n_estimators', low=300, high=3000, step=300),
        'learning_rate': trial.suggest_float(name='learning_rate', low=0.01, high=0.15, log=True),
        'max_depth': trial.suggest_int(name='max_depth', low=2, high=6),
        'min_child_weight': trial.suggest_float(name='min_child_weight', low=1.0, high=30.0, log=True),
        'subsample': trial.suggest_float(name='subsample', low=0.6, high=1.0),
        'colsample_bytree': trial.suggest_float(name='colsample_bytree', low=0.6, high=1.0),
        'reg_alpha': trial.suggest_categorical(name='reg_alpha', choices=[0.0, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0]),
        'reg_lambda': trial.suggest_float(name='reg_lambda', low=1e-3, high=100.0, log=True),
        'gamma': trial.suggest_categorical(name='gamma', choices=[0.0, 1e-4, 1e-3, 1e-2, 0.05, 0.1]),
    }

    estimator = clone(model).set_params(**params)

    pipeline = make_pipeline(
        preprocessing,
        FunctionTransformer(
            to_gpu,
            validate=False,
        ).set_output(transform='default'),
        estimator,
    )

    score = -cross_val_score(
        pipeline,
        X_train,
        np.log1p(y_train),      # log1p(y) = log(1+y), since the competition compares log vs log.
        cv=KFold(
            n_splits=5,
            shuffle=True,
            random_state=seed,
        ),
        scoring='neg_root_mean_squared_error',
        n_jobs=1,
        error_score='raise',
    ).mean()

    return score

# 2. Study
study = optuna.create_study(
    storage=None,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=5,
        seed=seed,
    ),
    pruner=optuna.pruners.NopPruner(),
)

# 3. Optimize
study.optimize(
    func=objective,
    n_trials=20,
    timeout=None,
    n_jobs=1,
    show_progress_bar=True,
    gc_after_trial=True,
)

print(f'Best CV RMSE: {study.best_value:.6f}')
pprint(study.best_params)

# Best model
best_estimator = make_pipeline(
    clone(preprocessing),
    FunctionTransformer(
        to_gpu,
        validate=False,
    ).set_output(transform='default'),
    clone(model).set_params(**study.best_params),
)

best_estimator.fit(X_train, np.log1p(y_train))
y_pred = np.expm1(best_estimator.predict(df_test[X_train.columns]).astype(float))

# Submit
submit = pd.DataFrame(
    {
        'Id': df_test['Id'],
        'SalePrice': y_pred,
    }
)
# submit.to_csv('./xgb.csv', index=False)

# 5. Ensemble - Voting

## 1) xgb/ridge/catboost

In [ ]:
import cupy as cp
from catboost import CatBoostRegressor
from sklearn.base import clone
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.model_selection import cross_val_predict


# Helpers
def to_gpu(X):
    return cp.asarray(np.asarray(X), dtype=cp.float32)

def prepare_catboost(X):
    X = X.copy()
    X[cols_cat] = X[cols_cat].astype(object).fillna('__MISSING__').astype(str)
    return X

# Base estimators (with preprocessing)
pipe_xgb = make_pipeline(
    clone(preprocessing),
    clone(model).set_params(    # manual selection from 4. HPO.
        n_estimators=3_000,
        learning_rate=0.011848263726052681,
        max_depth=4,
        min_child_weight=2.0841441217786407,
        subsample=0.6526520058451307,
        colsample_bytree=0.6195851590057587,
        reg_alpha=0.1,
        reg_lambda=5.784000925359463,
        gamma=0.0001,
    ),
)

pipe_ridge = make_pipeline(
    ColumnTransformer(
        [
            ('pipe_num', clone(prep_num), cols_num),
            ('pipe_cat', make_pipeline(
                SimpleImputer(strategy='constant', fill_value='__MISSING__'),
                OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            ), cols_cat),
        ]
    ),
    RidgeCV(
        alphas=np.logspace(-2, 3, 13),
        scoring='neg_mean_squared_error',
    ),
)

pipe_catboost = make_pipeline(
    FunctionTransformer(
        prepare_catboost,
        validate=False,
    ).set_output(transform='default'),
    CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=5.0,
        loss_function='RMSE',
        cat_features=tuple(cols_cat),
        random_seed=seed,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
    ),
)

voting = VotingRegressor(
    estimators=[
        ('xgb', pipe_xgb),
        ('ridge', pipe_ridge),
        ('catboost', pipe_catboost),
    ],
    weights=[1, 1, 1],
    n_jobs=1,
)

# Cross validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=seed,
)
names = ['xgb', 'ridge', 'catboost', 'voting']
y_log = np.log1p(y_train)

oof_pred = pd.DataFrame(
    {
        name: cross_val_predict(
            estimator,
            X_train,
            y_log,
            cv=cv,
            n_jobs=1,
            verbose=10,
        )
        for name, estimator in voting.estimators
    },
    index=X_train.index,
)

oof_pred['voting'] = np.average(
    oof_pred[names[:-1]],
    axis=1,
    weights=voting.weights,
)

fold_scores = []

for _, valid_idx in cv.split(X_train):
    errors = (
        oof_pred.iloc[valid_idx].to_numpy()
        - y_log.iloc[valid_idx].to_numpy()[:, None]
    )
    fold_scores.append(np.sqrt(np.mean(errors ** 2, axis=0)))

cv_results = pd.DataFrame(fold_scores, columns=names).agg(['mean', 'std']).T
cv_results.columns = ['CV mean RMSE', 'CV std']
cv_results['OOF RMSE'] = np.sqrt(oof_pred.sub(y_log, axis=0).pow(2).mean())

print('\nRMSE on log1p(SalePrice):')
print(cv_results.to_string(float_format='{:.6f}'.format))

print('\nOOF residual correlations:')
print(oof_pred[names[:-1]].sub(y_log, axis=0).corr().round(3))

print('\nRefitting on all training data...', flush=True)
voting.fit(X_train, y_log)

y_pred = np.expm1(voting.predict(df_test[X_train.columns]).astype(float))

if not np.isfinite(y_pred).all() or (y_pred <= 0).any():
    raise ValueError('Predicted prices must be finite and positive.')

# Save for submission
submit = pd.DataFrame(
    {
        'Id': df_test['Id'],
        'SalePrice': y_pred,
    }
)
# submit.to_csv('./voting.csv', index=False)

print(f"Selected Ridge alpha: {voting.named_estimators_['ridge'][-1].alpha_:.6g}")
print('Saved: voting.csv')

Fold 1/5: xgb=0.133045, ridge=0.137858, catboost=0.127536, voting=0.127570
Fold 2/5: xgb=0.110978, ridge=0.130119, catboost=0.112269, voting=0.111682
Fold 3/5: xgb=0.144454, ridge=0.225581, catboost=0.151609, voting=0.167925
Fold 4/5: xgb=0.126318, ridge=0.126700, catboost=0.125252, voting=0.119141
Fold 5/5: xgb=0.108286, ridge=0.115016, catboost=0.105861, voting=0.102142

RMSE on log1p(SalePrice):
          CV mean RMSE   CV std  OOF RMSE
xgb           0.124616 0.015167  0.125352
ridge         0.147055 0.044661  0.152384
catboost      0.124505 0.017619  0.125499
voting        0.125692 0.025401  0.127729

OOF residual correlations:
            xgb  ridge  catboost
xgb       1.000  0.806     0.932
ridge     0.806  1.000     0.839
catboost  0.932  0.839     1.000

Refitting on all training data...
Selected Ridge alpha: 21.5443
Saved: voting.csv


## 2) Inspection

In [ ]:
base_names = ['xgb', 'ridge', 'catboost']
check_names = base_names + ['voting']

if not oof_pred.index.equals(X_train.index) or not y_train.index.equals(X_train.index):
    raise ValueError('X_train, y_train, and oof_pred must retain their original row order.')

fold_id = pd.Series(0, index=X_train.index, name='fold')
for fold, (_, valid_idx) in enumerate(cv.split(X_train), start=1):
    fold_id.iloc[valid_idx] = fold

pred_check = oof_pred[check_names].astype(float).copy()
pred_check['trees'] = pred_check[['xgb', 'catboost']].mean(axis=1)
pred_check['ridge_10pct'] = 0.9 * pred_check['trees'] + 0.1 * pred_check['ridge']

if not np.isfinite(pred_check.to_numpy()).all():
    raise ValueError('OOF predictions must be complete and finite.')

residual = pred_check.sub(np.log1p(y_train), axis=0)
squared_error = residual.pow(2)

fold_rmse = np.sqrt(squared_error.groupby(fold_id).mean())
score_check = fold_rmse.T.rename(columns=lambda x: f'fold_{x}')
score_check['CV mean'] = fold_rmse.mean()
score_check['OOF RMSE'] = np.sqrt(squared_error.mean())

print('1. All-row comparison: RMSE on log1p(SalePrice)')
print(score_check.to_string(float_format='{:.6f}'.format))

concentration = []

for fold in sorted(fold_id.unique()):
    for name in check_names:
        error = residual.loc[fold_id.eq(fold), name].abs()
        loss = error.pow(2)
        total = loss.sum()

        concentration.append({
            'fold': fold,
            'model': name,
            'median_AE': error.median(),
            'p95_AE': error.quantile(0.95),
            'max_AE': error.max(),
            'top1_SSE_pct': 100 * loss.nlargest(1).sum() / total if total else 0,
            'top5_SSE_pct': 100 * loss.nlargest(5).sum() / total if total else 0,
        })

concentration = pd.DataFrame(concentration).set_index(['fold', 'model'])

print('\n2. Error concentration within each fold')
print(concentration.to_string(float_format='{:.4f}'.format))

diagnostics = pd.concat(
    [
        df_train.loc[X_train.index].copy(),
        fold_id,
        np.expm1(pred_check).add_prefix('pred_'),
        residual.add_prefix('residual_'),
        squared_error.add_prefix('squared_error_'),
    ],
    axis=1,
)
diagnostics['shared_direction'] = np.select(
    [
        residual[base_names].gt(0).all(axis=1),
        residual[base_names].lt(0).all(axis=1),
    ],
    ['over', 'under'],
    default='mixed',
)
diagnostics['ridge_damage'] = squared_error['voting'] - squared_error['trees']
diagnostics['voting_fold_SSE_pct'] = (
    100 * squared_error['voting']
    / squared_error['voting'].groupby(fold_id).transform('sum').replace(0, np.nan)
)

inspect_idx = []
for name in check_names:
    inspect_idx.extend(
        squared_error.loc[fold_id.eq(3), name].nlargest(3).index
    )
inspect_idx.extend(squared_error['voting'].nlargest(5).index)

worst_cases = diagnostics.loc[pd.Index(inspect_idx).unique()].sort_values(
    ['fold', 'squared_error_voting'],
    ascending=[True, False],
)

print('\n3. Fold 3 top errors per model, plus the overall top voting errors')
print(worst_cases[
    [
        'Id', 'fold', 'SalePrice', 'pred_xgb', 'pred_ridge',
        'pred_catboost', 'pred_voting', 'shared_direction',
        'voting_fold_SSE_pct',
    ]
].to_string(index=False, float_format='{:.2f}'.format))

feature_cols = [
    col for col in [
        'Neighborhood', 'GrLivArea', 'TotalBsmtSF', 'OverallQual',
        'OverallCond', 'YearBuilt', 'YearRemodAdd', 'SaleCondition',
    ]
    if col in X_train.columns
]

print('\n4. Original features of the same selected rows')
print(worst_cases[
    ['Id', 'fold', 'SalePrice'] + feature_cols
].to_string(index=False))

print('\n5. Rows where equal-weight Ridge increases loss most versus trees')
print(diagnostics.nlargest(5, 'ridge_damage')[
    [
        'Id', 'fold', 'SalePrice', 'pred_ridge', 'pred_trees',
        'pred_voting', 'ridge_damage',
    ]
].to_string(index=False, float_format='{:.4f}'.format))

print('\nSaved: ensemble_diagnostics.csv')
print('No rows were removed, no models were refitted, and voting.csv is unchanged.')

1. All-row comparison: RMSE on log1p(SalePrice)
fold          fold_1   fold_2   fold_3   fold_4   fold_5  CV mean  OOF RMSE
xgb         0.133045 0.110978 0.144454 0.126318 0.108286 0.124616  0.125352
ridge       0.137858 0.130119 0.225581 0.126700 0.115016 0.147055  0.152384
catboost    0.127536 0.112269 0.151609 0.125252 0.105861 0.124505  0.125499
voting      0.127570 0.111682 0.167925 0.119141 0.102142 0.125692  0.127729
trees       0.128646 0.109496 0.146323 0.122967 0.104272 0.122341  0.123242
ridge_10pct 0.127876 0.109600 0.152079 0.121297 0.103016 0.122774  0.123950

2. Error concentration within each fold
               median_AE  p95_AE  max_AE  top1_SSE_pct  top5_SSE_pct
fold model                                                          
1    xgb          0.0591  0.2621  0.7701       11.4734       35.3761
     ridge        0.0681  0.2438  0.7711       10.7144       33.2303
     catboost     0.0577  0.2652  0.7621       12.2299       34.2367
     voting       0.0524  0.2592  

## 3) Log Ridge

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import cross_val_predict


# Log transformation for 19 area-related features
log_cols = [
    'LotFrontage', 'LotArea', 'MasVnrArea',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
    'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch',
    '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal',
]

def log_ridge_features(X):
    X = X.copy()
    X[log_cols] = np.log1p(X[log_cols])
    return X

# Preprocessing pipeline
pipe_ridge_log = make_pipeline(
    FunctionTransformer(
        log_ridge_features,
        validate=False,
    ).set_output(transform='default'),
    clone(pipe_ridge),
)

if not oof_pred.index.equals(X_train.index) or not y_train.index.equals(X_train.index):
    raise ValueError('X_train, y_train, and oof_pred must retain their original row order.')

y_log = np.log1p(y_train)
fold_id_log = pd.Series(0, index=X_train.index)

oof_compare = oof_pred[['xgb', 'ridge', 'catboost', 'voting']].copy()
oof_compare['trees'] = oof_compare[['xgb', 'catboost']].mean(axis=1)
oof_compare['ridge_log'] = cross_val_predict(
    pipe_ridge_log,
    X_train,
    y_log,
    cv=cv,
    n_jobs=1,
    verbose=10,
)

for fold, (_, valid_idx) in enumerate(cv.split(X_train), start=1):
    fold_id_log.iloc[valid_idx] = fold

oof_compare['voting_logridge'] = oof_compare[
    ['xgb', 'ridge_log', 'catboost']
].mean(axis=1)

squared_error_log = oof_compare.sub(y_log, axis=0).pow(2)
fold_rmse_log = np.sqrt(squared_error_log.groupby(fold_id_log).mean())

results_log = fold_rmse_log.T.rename(columns=lambda x: f'fold_{x}')
results_log['CV mean'] = fold_rmse_log.mean()
results_log['OOF RMSE'] = np.sqrt(squared_error_log.mean())

print('\nAll-row comparison: RMSE on log1p(SalePrice)')
print(results_log.to_string(float_format='{:.6f}'.format))

case_idx = df_train.index[df_train['Id'].isin([524, 1299])]
case_results = pd.concat(
    [
        df_train.loc[case_idx, ['Id', 'SalePrice']],
        np.expm1(oof_compare.loc[
            case_idx,
            ['ridge', 'ridge_log', 'trees', 'voting_logridge'],
        ]).add_prefix('pred_'),
    ],
    axis=1,
)

print('\nOOF predictions for the two inspected houses')
print(case_results.to_string(index=False, float_format='{:.2f}'.format))

print('\nRefitting revised Ridge on all training data...', flush=True)
ridge_log_model = clone(pipe_ridge_log)
ridge_log_model.fit(X_train, y_log)

X_test = df_test[X_train.columns]

test_pred_log = pd.DataFrame(
    {
        'xgb': voting.named_estimators_['xgb'].predict(X_test),
        'ridge_log': ridge_log_model.predict(X_test),
        'catboost': voting.named_estimators_['catboost'].predict(X_test),
    },
    index=df_test.index,
).astype(float)

submission_pred_log = pd.DataFrame(
    {
        'voting_trees': test_pred_log[['xgb', 'catboost']].mean(axis=1),
        'voting_logridge': test_pred_log.mean(axis=1),
    }
)

for name, pred_log in submission_pred_log.items():
    prices = np.expm1(pred_log)

    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError(f'{name} contains invalid predicted prices.')

    submission = pd.DataFrame(
        {
            'Id': df_test['Id'],
            'SalePrice': prices,
        }
    )
    submission.to_csv(f'./{name}.csv', index=False)

    print(f'Saved: {name}.csv')

oof_compare.assign(
    Id=df_train.loc[X_train.index, 'Id'],
    SalePrice=y_train,
    fold=fold_id_log,
).to_csv('./ridge_log_oof.csv', index=False)

print(f'Selected Ridge alpha: {ridge_log_model[-1][-1].alpha_:.6g}')
print('Saved: ridge_log_oof.csv')

Fold 1/5: ridge_log=0.131149
Fold 2/5: ridge_log=0.120008
Fold 3/5: ridge_log=0.169902
Fold 4/5: ridge_log=0.121209
Fold 5/5: ridge_log=0.111467

All-row comparison: RMSE on log1p(SalePrice)
                  fold_1   fold_2   fold_3   fold_4   fold_5  CV mean  OOF RMSE
xgb             0.133045 0.110978 0.144454 0.126318 0.108286 0.124616  0.125352
ridge           0.137858 0.130119 0.225581 0.126700 0.115016 0.147055  0.152384
catboost        0.127536 0.112269 0.151609 0.125252 0.105861 0.124505  0.125499
voting          0.127570 0.111682 0.167925 0.119141 0.102142 0.125692  0.127729
trees           0.128646 0.109496 0.146323 0.122967 0.104272 0.122341  0.123242
ridge_log       0.131149 0.120008 0.169902 0.121209 0.111467 0.130747  0.132352
voting_logridge 0.126399 0.109310 0.151140 0.118710 0.101815 0.121475  0.122660

OOF predictions for the two inspected houses
  Id  SalePrice  pred_ridge  pred_ridge_log  pred_trees  pred_voting_logridge
 524     184750  1069684.21       684989.64  

## 4) SVR

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_predict


pipe_svr = make_pipeline(
    clone(pipe_ridge_log[0]),
    clone(pipe_ridge[0]),
    SVR(
        kernel='rbf',
        C=10.0,
        epsilon=0.01,
        gamma=0.001,
    ),
)

if not oof_compare.index.equals(X_train.index) or not y_train.index.equals(X_train.index):
    raise ValueError('X_train, y_train, and OOF predictions must retain their original row order.')

if not test_pred_log.index.equals(df_test.index):
    raise ValueError('Test predictions and df_test must retain their original row order.')

y_log = np.log1p(y_train)
fold_id_svr = pd.Series(0, index=X_train.index)
oof_svr = oof_compare[['xgb', 'ridge_log', 'catboost', 'voting_logridge']].copy()
oof_svr['svr'] = np.nan

oof_svr['svr'] = cross_val_predict(
    pipe_svr,
    X_train,
    y_log,
    cv=cv,
    n_jobs=1,
    verbose=10,
)

for fold, (_, valid_idx) in enumerate(cv.split(X_train), start=1):
    fold_id_svr.iloc[valid_idx] = fold

if not np.array_equal(fold_id_svr.to_numpy(), fold_id_log.to_numpy()):
    raise ValueError('Use the same CV splits as the previous log-Ridge experiment.')

oof_svr['voting_svr'] = oof_svr[['xgb', 'svr', 'catboost']].mean(axis=1)
oof_svr['voting_four'] = oof_svr[['xgb', 'ridge_log', 'catboost', 'svr']].mean(axis=1)

if not np.isfinite(oof_svr.to_numpy()).all():
    raise ValueError('OOF predictions must be complete and finite.')

error_svr = oof_svr.sub(y_log, axis=0)
fold_rmse_svr = np.sqrt(error_svr.pow(2).groupby(fold_id_svr).mean())

results_svr = fold_rmse_svr.T.rename(columns=lambda x: f'fold_{x}')
results_svr['CV mean'] = fold_rmse_svr.mean()
results_svr['OOF RMSE'] = np.sqrt(error_svr.pow(2).mean())

print('\nAll-row comparison: RMSE on log1p(SalePrice)')
print(results_svr.to_string(float_format='{:.6f}'.format))

print('\nOOF residual correlations:')
print(error_svr[['xgb', 'ridge_log', 'catboost', 'svr']].corr().round(3))

print('\nRefitting SVR on all training data...', flush=True)
svr_model = clone(pipe_svr)
svr_model.fit(X_train, y_log)

test_svr = test_pred_log[['xgb', 'catboost']].copy()
test_svr['svr'] = svr_model.predict(df_test[X_train.columns])
y_pred_svr = np.expm1(test_svr.mean(axis=1))

if not np.isfinite(y_pred_svr).all() or (y_pred_svr <= 0).any():
    raise ValueError('Predicted prices must be finite and positive.')

submit_svr = pd.DataFrame(
    {
        'Id': df_test['Id'],
        'SalePrice': y_pred_svr,
    }
)
submit_svr.to_csv('./voting_svr.csv', index=False)

oof_svr.assign(
    Id=df_train.loc[X_train.index, 'Id'],
    SalePrice=y_train,
    fold=fold_id_svr,
).to_csv('./svr_oof.csv', index=False)

print('Saved: voting_svr.csv')
print('Saved: svr_oof.csv')

Fold 1/5: svr=0.131137
Fold 2/5: svr=0.119689
Fold 3/5: svr=0.170624
Fold 4/5: svr=0.114555
Fold 5/5: svr=0.099832

All-row comparison: RMSE on log1p(SalePrice)
                  fold_1   fold_2   fold_3   fold_4   fold_5  CV mean  OOF RMSE
xgb             0.133045 0.110978 0.144454 0.126318 0.108286 0.124616  0.125352
ridge_log       0.131149 0.120008 0.169902 0.121209 0.111467 0.130747  0.132352
catboost        0.127536 0.112269 0.151609 0.125252 0.105861 0.124505  0.125499
voting_logridge 0.126399 0.109310 0.151140 0.118710 0.101815 0.121475  0.122660
svr             0.131137 0.119689 0.170624 0.114555 0.099832 0.127167  0.129402
voting_svr      0.126528 0.108979 0.151573 0.116904 0.098370 0.120471  0.121823
voting_four     0.125960 0.109723 0.154689 0.116027 0.098832 0.121046  0.122526

OOF residual correlations:
             xgb  ridge_log  catboost    svr
xgb        1.000      0.850     0.932  0.852
ridge_log  0.850      1.000     0.871  0.935
catboost   0.932      0.871     1.00

# 6. Clipping

In [ ]:
clip_target = np.log1p(y_train)
clip_oof = oof_compare['voting_logridge'].copy()
clip_test = test_pred_log[['xgb', 'ridge_log', 'catboost']].mean(axis=1)

if not clip_oof.index.equals(y_train.index) or not fold_id_log.index.equals(y_train.index):
    raise ValueError('OOF predictions, targets, and fold IDs must retain their original row order.')

if not clip_test.index.equals(df_test.index):
    raise ValueError('Test predictions and df_test must retain their original row order.')

if not np.isfinite(clip_oof).all() or not np.isfinite(clip_test).all():
    raise ValueError('Predictions must be complete and finite.')

baseline_loss = (clip_oof - clip_target).pow(2)
inspected = df_train.loc[clip_oof.index, 'Id'].isin([524, 1299])
clip_rows = []

for upper in [np.inf, 700_000, 600_000, 500_000, 400_000]:
    pred = np.minimum(clip_oof, np.log1p(upper))
    loss = (pred - clip_target).pow(2)
    gain = baseline_loss - loss
    fold_rmse = np.sqrt(loss.groupby(fold_id_log).mean())

    clip_rows.append({
        'cap': 'None' if np.isinf(upper) else f'{upper:,}',
        'CV mean': fold_rmse.mean(),
        'OOF RMSE': np.sqrt(loss.mean()),
        'OOF changed': int((pred < clip_oof).sum()),
        'helped': int((gain > 1e-12).sum()),
        'hurt': int((gain < -1e-12).sum()),
        'SSE gain: two IDs': gain[inspected].sum(),
        'SSE gain: other rows': gain[~inspected].sum(),
        'test changed': int((clip_test > np.log1p(upper)).sum()),
    })

clip_results = pd.DataFrame(clip_rows).set_index('cap')

print('Upper clipping of the log-Ridge ensemble:')
print(clip_results.to_string(float_format='{:.6f}'.format))
print('\nPositive SSE gain means lower squared log-error.')

upper = 600_000
capped_prices = np.expm1(np.minimum(clip_test, np.log1p(upper)))
changed = clip_test > np.log1p(upper)

if not np.isfinite(capped_prices).all() or (capped_prices <= 0).any():
    raise ValueError('Predicted prices must be finite and positive.')

affected_test = pd.DataFrame(
    {
        'Id': df_test['Id'],
        'original_price': np.expm1(clip_test),
        'capped_price': capped_prices,
    }
)

print(f'\nTest predictions affected by the ${upper:,} cap:')
print(affected_test.loc[changed].to_string(
    index=False,
    float_format='{:.2f}'.format,
))

if changed.any():
    submit_clip = pd.DataFrame(
        {
            'Id': df_test['Id'],
            'SalePrice': capped_prices,
        }
    )
    filename = f'./voting_logridge_cap{upper}.csv'
    submit_clip.to_csv(filename, index=False)
    print(f'\nSaved: {filename}')
else:
    print('\nNo test predictions changed; no new submission was saved.')

Upper clipping of the log-Ridge ensemble:
         CV mean  OOF RMSE  OOF changed  helped  hurt  SSE gain: two IDs  SSE gain: other rows  test changed
cap                                                                                                         
None    0.121475  0.122660            0       0     0           0.000000              0.000000             0
700,000 0.121475  0.122660            0       0     0           0.000000              0.000000             0
600,000 0.120704  0.121719            2       2     0           0.335661              0.000000             0
500,000 0.118943  0.119617            4       2     2           1.180666             -0.104251             2
400,000 0.119497  0.119988           23       6    17           2.033915             -1.087494            31

Positive SSE gain means lower squared log-error.

Test predictions affected by the $600,000 cap:
Empty DataFrame
Columns: [Id, original_price, capped_price]
Index: []

No test predictions change

# 7. Huber

In [ ]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


robust_prep = make_pipeline(
    clone(pipe_ridge_log[0]),
    clone(pipe_ridge[0]),
).set_output(transform='default')

robust_models = {
    'ridge_strong': Ridge(alpha=100.0),
    'huber': TransformedTargetRegressor(
        regressor=HuberRegressor(
            epsilon=1.35,
            alpha=1.0,
            max_iter=5000,
        ),
        transformer=StandardScaler(with_std=False).set_output(transform='default'),
    ),
}

if not oof_compare.index.equals(X_train.index) or not y_train.index.equals(X_train.index):
    raise ValueError('Training rows and OOF predictions must retain their original order.')

if not fold_id_log.index.equals(X_train.index) or not test_pred_log.index.equals(df_test.index):
    raise ValueError('Fold IDs and test predictions must retain their original row order.')

y_log = np.log1p(y_train)
oof_robust = oof_compare[['xgb', 'ridge_log', 'catboost', 'voting_logridge']].copy()

for name, regressor in robust_models.items():
    oof_robust[name] = cross_val_predict(
        make_pipeline(
            clone(robust_prep),
            clone(regressor),
        ),
        X_train,
        y_log,
        cv=cv,
        n_jobs=1,
        verbose=10,
    )

for name in robust_models:
    oof_robust[f'voting_{name}'] = oof_robust[['xgb', name, 'catboost']].mean(axis=1)

if not np.isfinite(oof_robust.to_numpy()).all():
    raise ValueError('OOF predictions must be complete and finite.')

loss_robust = oof_robust.sub(y_log, axis=0).pow(2)
fold_rmse_robust = np.sqrt(loss_robust.groupby(fold_id_log).mean())
results_robust = fold_rmse_robust.T.rename(columns=lambda x: f'fold_{x}')
results_robust['CV mean'] = fold_rmse_robust.mean()
results_robust['OOF RMSE'] = np.sqrt(loss_robust.mean())

print('\nAll-row comparison: RMSE on log1p(SalePrice)')
print(results_robust.to_string(float_format='{:.6f}'.format))

print('\nRefitting Huber on all training data...', flush=True)
pipe_huber = make_pipeline(
    clone(robust_prep),
    clone(robust_models['huber']),
)
pipe_huber.fit(X_train, y_log)

pred_huber_log = pipe_huber.predict(df_test[X_train.columns])
pred_vote_log = (
    test_pred_log['xgb'] + test_pred_log['catboost'] + pred_huber_log
) / 3
prices = np.expm1(pred_vote_log)

if not np.isfinite(prices).all() or (prices <= 0).any():
    raise ValueError('Predicted prices must be finite and positive.')

submit_huber = pd.DataFrame(
    {
        'Id': df_test['Id'],
        'SalePrice': prices,
    }
)
submit_huber.to_csv('./voting_huber.csv', index=False)

oof_robust.assign(
    Id=df_train.loc[X_train.index, 'Id'],
    SalePrice=y_train,
    fold=fold_id_log,
).to_csv('./robust_oof.csv', index=False)

print('Saved: voting_huber.csv and robust_oof.csv')

Fold 1/5: ridge_strong=0.135614, huber=0.126418
Fold 2/5: ridge_strong=0.119788, huber=0.122931
Fold 3/5: ridge_strong=0.170949, huber=0.174203
Fold 4/5: ridge_strong=0.125179, huber=0.132685
Fold 5/5: ridge_strong=0.112179, huber=0.113678

All-row comparison: RMSE on log1p(SalePrice)
                      fold_1   fold_2   fold_3   fold_4   fold_5  CV mean  OOF RMSE
xgb                 0.133045 0.110978 0.144454 0.126318 0.108286 0.124616  0.125352
ridge_log           0.131149 0.120008 0.169902 0.121209 0.111467 0.130747  0.132352
catboost            0.127536 0.112269 0.151609 0.125252 0.105861 0.124505  0.125499
voting_logridge     0.126399 0.109310 0.151140 0.118710 0.101815 0.121475  0.122660
ridge_strong        0.135614 0.119788 0.170949 0.125179 0.112179 0.132742  0.134326
huber               0.126418 0.122931 0.174203 0.132685 0.113678 0.133983  0.135623
voting_ridge_strong 0.127743 0.109461 0.151657 0.120435 0.101752 0.122210  0.123417
voting_huber        0.123143 0.108052 0.15

# 8. Data Pruning

- With-vs-without filtering
  - Exclude samples in the estimator's training that makes other predictions worse.

In [19]:
from sklearn.base import clone
from sklearn.model_selection import KFold


if not oof_compare.index.equals(X_train.index) or not y_train.index.equals(X_train.index):
    raise ValueError('Training rows and OOF predictions must retain their original order.')

if not fold_id_log.index.equals(X_train.index) or not test_pred_log.index.equals(df_test.index):
    raise ValueError('Fold IDs and test predictions must retain their original row order.')

keep_ridge = ~((X_train['GrLivArea'] > 4000) & (y_train < 300000))
y_log = np.log1p(y_train)
validation_rows = []
validation_oof = []

print('Excluded from Ridge fitting only:')
print(df_train.loc[~keep_ridge, ['Id', 'GrLivArea', 'SalePrice']].to_string(index=False))

for split_seed in [seed, seed + 1, seed + 2]:
    splitter = cv if split_seed == seed else KFold(
        n_splits=5,
        shuffle=True,
        random_state=split_seed,
    )
    predictions = np.empty((len(X_train), 2))
    fold_labels = np.zeros(len(X_train), dtype=int)
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X_train), start=1):
        X_fit = X_train.iloc[train_idx]
        X_valid = X_train.iloc[valid_idx]
        y_fit = y_log.iloc[train_idx]
        filtered_idx = train_idx[keep_ridge.iloc[train_idx].to_numpy()]
        fold_labels[valid_idx] = fold

        if split_seed == seed:
            if not fold_id_log.iloc[valid_idx].eq(fold).all():
                raise ValueError('The original CV splits have changed.')

            tree_sum = oof_compare.iloc[valid_idx][['xgb', 'catboost']].sum(axis=1).to_numpy()
            ridge_base = oof_compare['ridge_log'].iloc[valid_idx].to_numpy()
        else:
            tree_sum = np.zeros(len(valid_idx))

            for name in ['xgb', 'catboost']:
                estimator = clone(voting.named_estimators_[name])
                estimator.fit(X_fit, y_fit)
                tree_sum += estimator.predict(X_valid)

            estimator = clone(pipe_ridge_log)
            estimator.fit(X_fit, y_fit)
            ridge_base = estimator.predict(X_valid)

        if len(filtered_idx) == len(train_idx):
            ridge_filtered = ridge_base
        else:
            estimator = clone(pipe_ridge_log)
            estimator.fit(X_train.iloc[filtered_idx], y_log.iloc[filtered_idx])
            ridge_filtered = estimator.predict(X_valid)

        predictions[valid_idx, 0] = (tree_sum + ridge_base) / 3
        predictions[valid_idx, 1] = (tree_sum + ridge_filtered) / 3

        rmse = np.sqrt(np.mean(
            (predictions[valid_idx] - y_log.iloc[valid_idx].to_numpy()[:, None]) ** 2,
            axis=0,
        ))
        fold_scores.append(rmse)

        print(
            f'Seed {split_seed}, fold {fold}/5: '
            f'baseline={rmse[0]:.6f}, filtered={rmse[1]:.6f}',
            flush=True,
        )

    if not np.isfinite(predictions).all():
        raise ValueError('OOF predictions must be complete and finite.')

    scores = np.sqrt(np.mean(
        (predictions - y_log.to_numpy()[:, None]) ** 2,
        axis=0,
    ))
    fold_scores = np.asarray(fold_scores)

    validation_rows.append({
        'seed': split_seed,
        'baseline OOF': scores[0],
        'filtered OOF': scores[1],
        'difference': scores[1] - scores[0],
        'folds improved': int(
            (fold_scores[:, 1] < fold_scores[:, 0] - 1e-12).sum()
        ),
    })

    validation_oof.append(pd.DataFrame({
        'Id': df_train.loc[X_train.index, 'Id'].to_numpy(),
        'SalePrice': y_train.to_numpy(),
        'seed': split_seed,
        'fold': fold_labels,
        'baseline_log': predictions[:, 0],
        'filtered_log': predictions[:, 1],
    }))

filter_results = pd.DataFrame(validation_rows).set_index('seed')

print('\nAll-row OOF comparison; negative difference favors filtering:')
print(filter_results.to_string(float_format='{:.6f}'.format))

filter_results.to_csv('./trainfilter_validation.csv')
pd.concat(validation_oof, ignore_index=True).to_csv(
    './trainfilter_oof.csv',
    index=False,
)

if (filter_results['difference'] < 0).all():
    print('\nRefitting filtered Ridge...', flush=True)

    ridge_filtered_model = clone(pipe_ridge_log)
    ridge_filtered_model.fit(X_train.loc[keep_ridge], y_log.loc[keep_ridge])

    pred_log = (
        test_pred_log[['xgb', 'catboost']].sum(axis=1).to_numpy()
        + ridge_filtered_model.predict(df_test[X_train.columns])
    ) / 3
    prices = np.expm1(pred_log)

    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError('Predicted prices must be finite and positive.')

    submit_filter = pd.DataFrame(
        {
            'Id': df_test['Id'],
            'SalePrice': prices,
        }
    )
    submit_filter.to_csv('./voting_logridge_trainfilter.csv', index=False)

    print('Saved: voting_logridge_trainfilter.csv')
else:
    print(
        '\nFiltering did not improve every split seed. '
        'No new submission was written.'
    )

Excluded from Ridge fitting only:
  Id  GrLivArea  SalePrice
 524       4676     184750
1299       5642     160000
Seed 42, fold 1/5: baseline=0.126399, filtered=0.124520
Seed 42, fold 2/5: baseline=0.109310, filtered=0.106275
Seed 42, fold 3/5: baseline=0.151140, filtered=0.151140
Seed 42, fold 4/5: baseline=0.118710, filtered=0.116446
Seed 42, fold 5/5: baseline=0.101815, filtered=0.100650
Seed 43, fold 1/5: baseline=0.127834, filtered=0.127444
Seed 43, fold 2/5: baseline=0.106543, filtered=0.105777
Seed 43, fold 3/5: baseline=0.125323, filtered=0.121674
Seed 43, fold 4/5: baseline=0.110841, filtered=0.110056
Seed 43, fold 5/5: baseline=0.123876, filtered=0.125270
Seed 44, fold 1/5: baseline=0.121334, filtered=0.120033
Seed 44, fold 2/5: baseline=0.153453, filtered=0.153453
Seed 44, fold 3/5: baseline=0.121640, filtered=0.118825
Seed 44, fold 4/5: baseline=0.116476, filtered=0.115438
Seed 44, fold 5/5: baseline=0.093905, filtered=0.090957

All-row OOF comparison; negative difference 

# 9. Seed Averaging

In [25]:
from sklearn.base import clone
from sklearn.model_selection import KFold
from tqdm import tqdm


model_seeds = [seed, seed + 1, seed + 2]
split_seeds = [seed]

seed_pipelines = {
    name: clone(voting.named_estimators_[name])
    for name in ['xgb', 'catboost']
}
seed_pipelines['catboost'][-1].set_params(
    task_type='CPU',
    thread_count=-1,
)

def seed_predictions(pipeline, seed_parameter, X_fit, y_fit, X_predict, progress):
    prep = clone(pipeline[:-1])
    X_fit_prep = prep.fit_transform(X_fit, y_fit)
    X_predict_prep = prep.transform(X_predict)
    predictions = []

    for model_seed in model_seeds:
        estimator = clone(pipeline[-1]).set_params(**{seed_parameter: model_seed})
        estimator.fit(X_fit_prep, y_fit)
        predictions.append(estimator.predict(X_predict_prep))
        progress.update(1)

    return np.column_stack(predictions).astype(float)


if not X_train.index.equals(y_train.index):
    raise ValueError('Training features and targets must retain the same row order.')

y_seed = np.log1p(y_train)
keep_seed = ~((X_train['GrLivArea'] > 4000) & (y_train < 300000))
seed_columns = [f'model_seed_{value}' for value in model_seeds] + ['seed_average']
seed_rows = []
seed_oof = []

n_fits = len(split_seeds) * 5 * (1 + 2 * len(model_seeds))

with tqdm(total=n_fits, desc='CV training', unit='fit') as progress:
    for split_seed in split_seeds:
        splitter = KFold(n_splits=5, shuffle=True, random_state=split_seed)
        predictions = np.full((len(X_train), len(seed_columns)), np.nan)
        fold_labels = np.zeros(len(X_train), dtype=int)

        for fold, (train_idx, valid_idx) in enumerate(splitter.split(X_train), start=1):
            progress.set_postfix(split=split_seed, fold=fold)

            X_fit = X_train.iloc[train_idx]
            X_valid = X_train.iloc[valid_idx]
            y_fit = y_seed.iloc[train_idx]
            ridge_idx = train_idx[keep_seed.iloc[train_idx].to_numpy()]

            ridge = clone(pipe_ridge_log)
            ridge.fit(X_train.iloc[ridge_idx], y_seed.iloc[ridge_idx])
            ridge_pred = ridge.predict(X_valid)
            progress.update(1)

            xgb_pred = seed_predictions(
                seed_pipelines['xgb'], 'random_state',
                X_fit, y_fit, X_valid, progress,
            )
            cat_pred = seed_predictions(
                seed_pipelines['catboost'], 'random_seed',
                X_fit, y_fit, X_valid, progress,
            )

            individual = (xgb_pred + cat_pred + ridge_pred[:, None]) / 3
            predictions[valid_idx] = np.column_stack([
                individual,
                individual.mean(axis=1),
            ])
            fold_labels[valid_idx] = fold

            scores = np.sqrt(np.mean(
                (predictions[valid_idx] - y_seed.iloc[valid_idx].to_numpy()[:, None]) ** 2,
                axis=0,
            ))

            tqdm.write(
                f'Split {split_seed}, fold {fold}/5: '
                f'single_seed={scores[0]:.6f}, seed_average={scores[-1]:.6f}'
            )

        if not np.isfinite(predictions).all():
            raise ValueError('OOF predictions must be complete and finite.')

        scores = np.sqrt(np.mean(
            (predictions - y_seed.to_numpy()[:, None]) ** 2,
            axis=0,
        ))

        seed_rows.append({
            'split_seed': split_seed,
            **dict(zip(seed_columns, scores)),
        })

        seed_oof.append(pd.DataFrame(predictions, columns=seed_columns).assign(
            Id=df_train.loc[X_train.index, 'Id'].to_numpy(),
            SalePrice=y_train.to_numpy(),
            split_seed=split_seed,
            fold=fold_labels,
        ))

seed_results = pd.DataFrame(seed_rows).set_index('split_seed')

baseline_scores = pd.read_csv(
    './trainfilter_validation.csv'
).set_index('seed')['filtered OOF']

seed_results['vs_single'] = (
    seed_results['seed_average'] - seed_results[f'model_seed_{seed}']
)
seed_results['vs_original'] = (
    seed_results['seed_average'] - baseline_scores.reindex(seed_results.index)
)

print('\nOOF RMSE; negative differences favor seed averaging:')
print(seed_results.to_string(float_format='{:.6f}'.format))

seed_results.to_csv('./seed_average_validation.csv')
pd.concat(seed_oof, ignore_index=True).to_csv(
    './seed_average_oof.csv',
    index=False,
)

if (seed_results['vs_original'] < 0).all():
    print('\nRefitting the seed ensemble...', flush=True)

    X_test_seed = df_test[X_train.columns]

    with tqdm(total=1 + 2 * len(model_seeds), desc='Final training', unit='fit') as progress:
        ridge = clone(pipe_ridge_log)
        ridge.fit(X_train.loc[keep_seed], y_seed.loc[keep_seed])
        progress.update(1)

        xgb_test = seed_predictions(
            seed_pipelines['xgb'], 'random_state',
            X_train, y_seed, X_test_seed, progress,
        ).mean(axis=1)

        cat_test = seed_predictions(
            seed_pipelines['catboost'], 'random_seed',
            X_train, y_seed, X_test_seed, progress,
        ).mean(axis=1)

    prices = np.expm1(
        (xgb_test + cat_test + ridge.predict(X_test_seed)) / 3
    )

    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError('Predicted prices must be finite and positive.')

    pd.DataFrame({
        'Id': df_test['Id'],
        'SalePrice': prices,
    }).to_csv('./voting_seed_average.csv', index=False)

    print('Saved: voting_seed_average.csv')
else:
    print('\nSeed averaging did not beat the original ensemble on every split.')
    print('No new submission was written.')

CV training:  23%|██▎       | 8/35 [04:20<18:20, 40.76s/fit, fold=2, split=42]

Split 42, fold 1/5: single_seed=0.124520, seed_average=0.124978


CV training:  43%|████▎     | 15/35 [08:42<13:56, 41.82s/fit, fold=3, split=42]

Split 42, fold 2/5: single_seed=0.106275, seed_average=0.106097


CV training:  63%|██████▎   | 22/35 [13:05<09:07, 42.11s/fit, fold=4, split=42]

Split 42, fold 3/5: single_seed=0.151140, seed_average=0.151770


CV training:  83%|████████▎ | 29/35 [17:27<04:12, 42.07s/fit, fold=5, split=42]

Split 42, fold 4/5: single_seed=0.116446, seed_average=0.116727


CV training: 100%|██████████| 35/35 [21:53<00:00, 37.53s/fit, fold=5, split=42]

Split 42, fold 5/5: single_seed=0.100650, seed_average=0.101210

OOF RMSE; negative differences favor seed averaging:
            model_seed_42  model_seed_43  model_seed_44  seed_average  vs_single  vs_original
split_seed                                                                                   
42               0.121106       0.122115       0.121688      0.121473   0.000368     0.000368

Seed averaging did not beat the original ensemble on every split.
No new submission was written.


# 10. Optimal `n_threads` (for CatBoost)

In [28]:
from time import perf_counter
from sklearn.base import clone
from sklearn.model_selection import KFold
from tqdm import tqdm


train_idx, valid_idx = next(
    KFold(n_splits=5, shuffle=True, random_state=seed).split(X_train)
)

X_bench_train = X_train.iloc[train_idx]
X_bench_valid = X_train.iloc[valid_idx]
y_bench_train = np.log1p(y_train.iloc[train_idx])
y_bench_valid = np.log1p(y_train.iloc[valid_idx]).to_numpy()

benchmark_runs = [
    (repeat, threads)
    for repeat in range(1, 4)
    for threads in [4, 10, 20]
]
np.random.default_rng(seed).shuffle(benchmark_runs)

benchmark_rows = []
benchmark_predictions = {}

for repeat, threads in tqdm(benchmark_runs, desc='CPU benchmark', unit='fit'):
    candidate = clone(voting.named_estimators_['catboost'])
    candidate[-1].set_params(
        task_type='CPU',
        thread_count=threads,
        random_seed=seed,
        verbose=False,
    )

    started = perf_counter()
    candidate.fit(X_bench_train, y_bench_train)
    elapsed = perf_counter() - started

    pred = np.asarray(candidate.predict(X_bench_valid), dtype=float)
    benchmark_predictions[(threads, repeat)] = pred

    benchmark_rows.append({
        'threads': threads,
        'repeat': repeat,
        'fit_seconds': elapsed,
        'log_RMSE': np.sqrt(np.mean((pred - y_bench_valid) ** 2)),
    })
    tqdm.write(f'Threads={threads}, repeat={repeat}: {elapsed:.2f}s')

thread_benchmark = pd.DataFrame(benchmark_rows)
reference_pred = benchmark_predictions[(4, 1)]

thread_benchmark['max_prediction_delta'] = [
    np.max(np.abs(benchmark_predictions[(threads, repeat)] - reference_pred))
    for threads, repeat in thread_benchmark[['threads', 'repeat']].itertuples(
        index=False, name=None,
    )
]

thread_summary = thread_benchmark.groupby('threads').agg(
    median_seconds=('fit_seconds', 'median'),
    min_seconds=('fit_seconds', 'min'),
    max_seconds=('fit_seconds', 'max'),
    log_RMSE=('log_RMSE', 'mean'),
    max_prediction_delta=('max_prediction_delta', 'max'),
)
thread_summary['speedup_vs_4'] = (
    thread_summary.loc[4, 'median_seconds'] / thread_summary['median_seconds']
)

print('\nCPU thread comparison:')
print(thread_summary.to_string(float_format='{:.6g}'.format))

thread_benchmark.to_csv('./cpu_thread_benchmark.csv', index=False)
thread_summary.to_csv('./cpu_thread_summary.csv')

best_threads = int(thread_summary['median_seconds'].idxmin())
print(f'\nFastest measured setting: thread_count={best_threads}')

CPU benchmark:  11%|█         | 1/9 [01:24<11:12, 84.04s/fit]

Threads=4, repeat=2: 84.02s


CPU benchmark:  22%|██▏       | 2/9 [02:51<10:00, 85.79s/fit]

Threads=4, repeat=1: 87.00s


CPU benchmark:  33%|███▎      | 3/9 [04:09<08:13, 82.28s/fit]

...

Threads=10, repeat=1: 76.83s


CPU benchmark:  89%|████████▉ | 8/9 [10:58<01:22, 82.03s/fit]

Threads=20, repeat=2: 84.74s


CPU benchmark: 100%|██████████| 9/9 [12:22<00:00, 82.48s/fit]

Threads=20, repeat=3: 84.19s

CPU thread comparison:
         median_seconds  min_seconds  max_seconds  log_RMSE  max_prediction_delta  speedup_vs_4
threads                                                                                        
4               86.4801      84.0169      86.9969  0.127536                     0             1
10              76.8266       75.954      78.0706  0.127536                     0       1.12565
20               84.745       84.188      84.8577  0.127536                     0       1.02047

Fastest measured setting: thread_count=10
